# Insurance Premium Prediction with Explainable AI (XAI)

**Authors:** Hayeong Jae & Paweł Kazimiruk  
**Institution:** University of Warsaw  
**Course:** Explainable Artificial Intelligence — Capstone Project, May 2026

---

## Project Overview

Insurance companies increasingly rely on machine learning models to calculate customer premiums. However, these models are often opaque — nobody knows exactly which factors drive their decisions.

**Research Question:**  
> *Are AI-driven insurance premiums based on legitimate health risk factors, or do they depend on uncontrollable personal traits like sex and region — which would constitute algorithmic bias?*

**Approach:**  
We train an XGBoost regression model to predict insurance charges, then apply a comprehensive suite of XAI methods to interpret and explain the model's decision-making process.

**Dataset:** Medical Cost Personal Dataset — 1,338 records, 6 features, 1 continuous target (`charges`)

---

## Notebook Structure

| Section | Description |
|---|---|
| **Part 1 — EDA** | Exploratory data analysis and key statistical findings |
| **Part 2 — Modeling** | XGBoost model training, evaluation, and performance metrics |
| **Part 3 — XAI** | SHAP, LIME, PDP, Permutation Importance, Counterfactual Explanation |


---

# Part 1 — Exploratory Data Analysis (EDA)

## Problem Statement

Before building a predictive model, we explore the dataset to understand its structure, distributions, and — most importantly — the relationship between each feature and the target variable (`charges`).

The central question we investigate here:  
**Which features are most strongly associated with insurance premium costs?**

This EDA directly motivates our fairness analysis: if sex and region show little association with charges even in raw data, that is strong evidence that a well-trained model should not rely on them.


In [ ]:
# ================================
# 01. Imports & Setup
# ================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_theme(style="whitegrid")

df = pd.read_csv('../data/insurance.csv')

print("Shape:", df.shape)
print("\nFirst 5 rows:")
df.head()


## 1.1 Data Quality Check

We check for missing values, data types, and basic statistics to ensure the dataset is clean and ready for modelling.


In [ ]:
# ================================
# 02. Basic Info & Missing Values
# ================================

print("=== Data Types ===")
print(df.dtypes)

print("\n=== Missing Values ===")
print(df.isnull().sum())

print("\n=== Basic Statistics ===")
df.describe()


**Finding:** The dataset is clean — no missing values across all 1,338 records. The target variable `charges` ranges from ~\$1,122 to ~\$63,770, with a mean of ~\$13,270 and high standard deviation (~\$12,110), indicating significant variance that the model must explain.


## 1.2 Target Variable Distribution

We examine the distribution of `charges` to determine whether transformation is needed before modelling. Skewed targets can degrade regression performance.


In [ ]:
# ================================
# 03. Target Variable Distribution
# ================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['charges'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Insurance Charges')
axes[0].set_xlabel('Charges ($)')
axes[0].set_ylabel('Count')

axes[1].hist(np.log1p(df['charges']), bins=50, color='coral', edgecolor='white')
axes[1].set_title('Log-Transformed Charges')
axes[1].set_xlabel('log(Charges)')
axes[1].set_ylabel('Count')

plt.suptitle('Target Variable: Insurance Charges', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Skewness (original): {df['charges'].skew():.3f}")
print(f"Skewness (log):      {np.log1p(df['charges']).skew():.3f}")


**Finding:** The original `charges` distribution is heavily right-skewed (skewness = 1.516), indicating a small number of very high-cost customers. After log transformation, the skewness drops to -0.090 — nearly normal. We will apply `log1p` transformation to the target during modelling to improve model accuracy.


## 1.3 Feature Distributions

We visualise each feature's distribution to understand the data composition.


In [ ]:
# ================================
# 04. Feature Distributions
# ================================

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

axes[0,0].hist(df['age'], bins=30, color='steelblue', edgecolor='white')
axes[0,0].set_title('Age Distribution'); axes[0,0].set_xlabel('Age')

axes[0,1].hist(df['bmi'], bins=30, color='steelblue', edgecolor='white')
axes[0,1].set_title('BMI Distribution'); axes[0,1].set_xlabel('BMI')

axes[0,2].bar(df['children'].value_counts().index,
              df['children'].value_counts().values, color='steelblue')
axes[0,2].set_title('Number of Children'); axes[0,2].set_xlabel('Children')

axes[1,0].bar(df['sex'].value_counts().index,
              df['sex'].value_counts().values, color='coral')
axes[1,0].set_title('Sex Distribution')

axes[1,1].bar(df['smoker'].value_counts().index,
              df['smoker'].value_counts().values, color='coral')
axes[1,1].set_title('Smoker Distribution')

axes[1,2].bar(df['region'].value_counts().index,
              df['region'].value_counts().values, color='coral')
axes[1,2].set_title('Region Distribution')
axes[1,2].tick_params(axis='x', rotation=15)

plt.suptitle('Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


**Key Observations:**
- **Age** is fairly uniformly distributed between 18–64, with a spike at 18-19.
- **BMI** follows a roughly normal distribution centred around 30.
- **Smokers** represent approximately 20% of the dataset — a minority group with potentially outsized impact on premiums.
- **Sex** and **Region** are balanced, making them suitable for fairness analysis.


## 1.4 Feature vs. Charges Relationship

This is the most important section of the EDA. We examine how each feature relates to the target variable to form hypotheses for the XAI analysis.


In [ ]:
# ================================
# 05. Feature vs Charges Analysis
# ================================

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

axes[0,0].scatter(df['age'], df['charges'], alpha=0.4, color='steelblue')
axes[0,0].set_title('Age vs Charges'); axes[0,0].set_xlabel('Age'); axes[0,0].set_ylabel('Charges ($)')

axes[0,1].scatter(df['bmi'], df['charges'], alpha=0.4, color='steelblue')
axes[0,1].set_title('BMI vs Charges'); axes[0,1].set_xlabel('BMI')

axes[0,2].boxplot([df[df['children']==i]['charges'] for i in range(6)], tick_labels=range(6))
axes[0,2].set_title('Children vs Charges'); axes[0,2].set_xlabel('Number of Children')

axes[1,0].boxplot([df[df['smoker']=='no']['charges'], df[df['smoker']=='yes']['charges']],
                  tick_labels=['Non-smoker', 'Smoker'])
axes[1,0].set_title('Smoker vs Charges'); axes[1,0].set_ylabel('Charges ($)')

axes[1,1].boxplot([df[df['sex']=='male']['charges'], df[df['sex']=='female']['charges']],
                  tick_labels=['Male', 'Female'])
axes[1,1].set_title('Sex vs Charges')

axes[1,2].boxplot([df[df['region']==r]['charges'] for r in df['region'].unique()],
                  tick_labels=df['region'].unique())
axes[1,2].set_title('Region vs Charges')
axes[1,2].tick_params(axis='x', rotation=15)

plt.suptitle('Feature vs Insurance Charges', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


**Critical Findings:**
- **Smoker vs Charges:** The median charge for smokers is 3–4× higher than for non-smokers. Smoking is clearly the dominant predictor.
- **Age vs Charges:** Three visible clusters emerge — likely separating smokers and non-smokers — with a positive trend as age increases.
- **BMI vs Charges:** Higher BMI tends to correlate with higher charges, particularly above 30.
- **Sex vs Charges:** The distributions for male and female are nearly identical — strong evidence that sex should not materially affect premiums.
- **Region vs Charges:** All four regions show similar distributions — region appears to have minimal predictive value.


## 1.5 Correlation Analysis

We encode categorical variables and compute correlations with the target to quantify the direction and strength of each feature's relationship with charges.


In [ ]:
# ================================
# 06. Correlation Analysis
# ================================

df_encoded = df.copy()
df_encoded['sex'] = df_encoded['sex'].map({'male': 1, 'female': 0})
df_encoded['smoker'] = df_encoded['smoker'].map({'yes': 1, 'no': 0})
df_encoded = pd.get_dummies(df_encoded, columns=['region'], drop_first=True)

corr = df_encoded.corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5)
plt.title('Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("=== Correlation with Charges ===")
print(corr['charges'].sort_values(ascending=False))


**Correlation Summary:**

| Feature | Correlation with Charges | Interpretation |
|---|---|---|
| `smoker` | **0.787** | Very strong positive — smoking is the dominant factor |
| `age` | 0.299 | Moderate positive — older = higher premiums |
| `bmi` | 0.198 | Weak-moderate positive |
| `children` | 0.068 | Very weak |
| `sex` | 0.057 | **Negligible** — sex barely correlates with charges |
| `region` | -0.04 to 0.07 | **Negligible** — region barely correlates with charges |

This correlation analysis directly supports our fairness hypothesis: **sex and region have near-zero relationship with insurance costs**, even in raw data.


## 1.6 Business Insight Summary

We now quantify the average premium differences between groups to provide concrete numbers for the business analysis.


In [ ]:
# ================================
# 07. Key Business Insight — Average Charges
# ================================

print("=== Average Charges Comparison ===")
print(f"Smoker:     ${df[df['smoker']=='yes']['charges'].mean():,.0f}")
print(f"Non-smoker: ${df[df['smoker']=='no']['charges'].mean():,.0f}")
print(f"Difference: {df[df['smoker']=='yes']['charges'].mean() / df[df['smoker']=='no']['charges'].mean():.1f}x higher\n")

print(f"Male:       ${df[df['sex']=='male']['charges'].mean():,.0f}")
print(f"Female:     ${df[df['sex']=='female']['charges'].mean():,.0f}")
diff = abs(df[df['sex']=='male']['charges'].mean() - df[df['sex']=='female']['charges'].mean())
print(f"Difference: ${diff:,.0f}\n")

print("=== Average Charges by Region ===")
print(df.groupby('region')['charges'].mean().sort_values(ascending=False).apply(lambda x: f"${x:,.0f}"))


In [ ]:
# ================================
# 08. Business Summary Visualisation
# ================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

smoker_avg = df.groupby('smoker')['charges'].mean()
bars1 = axes[0].bar(['Non-Smoker', 'Smoker'], [smoker_avg['no'], smoker_avg['yes']],
                     color=['steelblue', 'crimson'], edgecolor='white')
axes[0].set_title('Smoking Status vs Avg Charges', fontweight='bold')
axes[0].set_ylabel('Average Charges ($)')
axes[0].bar_label(bars1, fmt='$%.0f', padding=5)
axes[0].set_ylim(0, 40000)

sex_avg = df.groupby('sex')['charges'].mean()
bars2 = axes[1].bar(['Male', 'Female'], [sex_avg['male'], sex_avg['female']],
                     color=['steelblue', 'steelblue'], edgecolor='white')
axes[1].set_title('Sex vs Avg Charges', fontweight='bold')
axes[1].set_ylabel('Average Charges ($)')
axes[1].bar_label(bars2, fmt='$%.0f', padding=5)
axes[1].set_ylim(0, 40000)

region_avg = df.groupby('region')['charges'].mean().sort_values(ascending=False)
bars3 = axes[2].bar(list(region_avg.index), list(region_avg.values),
                     color='steelblue', edgecolor='white')
axes[2].set_title('Region vs Avg Charges', fontweight='bold')
axes[2].set_ylabel('Average Charges ($)')
for bar, val in zip(bars3, region_avg.values):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 300,
                f'${val:,.0f}', ha='center', va='bottom', fontsize=9)
axes[2].set_ylim(0, 40000)
axes[2].tick_params(axis='x', rotation=15)

plt.suptitle('Key Insight: Smoking Dominates, Sex & Region Barely Matter',
             fontsize=13, fontweight='bold', color='darkred')
plt.tight_layout()
plt.show()


**EDA Conclusion:**

The exploratory analysis reveals a clear pattern:
- **Smoking** raises average premiums by **3.8×** (~\$32,050 vs \$8,434)
- **Sex** difference is only \$1,387 — practically negligible
- **Regional** variation spans just \$2,388 — also negligible

These findings directly inform our XAI analysis: if the model is well-calibrated, SHAP values should confirm that `smoker`, `age`, and `bmi` dominate, while `sex` and `region` contribute near zero.


---

# Part 2 — Model Development

## Approach & Model Selection

We train an **XGBoost Regressor** — an ensemble tree-based model that is powerful and widely used in industry, but inherently a **black box**. This makes it an ideal candidate for XAI interpretation.

**Why XGBoost?**
- Handles non-linear relationships and interactions between features naturally
- Robust to outliers and does not require feature scaling
- Strong out-of-the-box performance on tabular data
- As a black-box model, it necessitates XAI methods to explain its decisions

**Target transformation:**  
As identified in Part 1, `charges` is right-skewed. We apply `log1p` transformation during training and `expm1` to convert predictions back to dollar values.

**Preprocessing:**  
Categorical variables (`sex`, `smoker`, `region`) are encoded using label encoding and one-hot encoding.


In [ ]:
# ================================
# 01. Imports & Data Preparation
# ================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
import joblib, os, warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/insurance.csv')

df_model = df.copy()
df_model['sex'] = LabelEncoder().fit_transform(df_model['sex'])
df_model['smoker'] = LabelEncoder().fit_transform(df_model['smoker'])
df_model = pd.get_dummies(df_model, columns=['region'], drop_first=True)

X = df_model.drop('charges', axis=1)
y = np.log1p(df_model['charges'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Train set:", X_train.shape)
print("Test set: ", X_test.shape)
print("\nFeatures:", list(X.columns))


## 2.1 Model Training

We train XGBoost with carefully chosen hyperparameters. The 80/20 train-test split with `random_state=42` ensures reproducibility.


In [ ]:
# ================================
# 02. Model Training — XGBoost
# ================================

model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)
print("Model training complete.")


## 2.2 Model Evaluation

We evaluate performance on the held-out test set using three metrics:
- **R²**: Proportion of variance explained (higher is better, max = 1.0)
- **MAE**: Mean Absolute Error in USD (lower is better)
- **RMSE**: Root Mean Squared Error in USD (lower is better, more sensitive to large errors)

All predictions are converted back from log-scale to dollar values before evaluation.


In [ ]:
# ================================
# 03. Model Evaluation
# ================================

y_pred_log = model.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_test_orig = np.expm1(y_test)

rmse = np.sqrt(mean_squared_error(y_test_orig, y_pred))
mae = mean_absolute_error(y_test_orig, y_pred)
r2 = r2_score(y_test_orig, y_pred)

print("=== Model Performance ===")
print(f"R2 Score: {r2:.4f}")
print(f"RMSE:     ${rmse:,.0f}")
print(f"MAE:      ${mae:,.0f}")


**Results:**
- **R² = 0.848** — The model explains ~85% of the variance in insurance premiums. This is a strong result for a real-world tabular dataset.
- **MAE = \$2,402** — On average, predictions are within \$2,402 of the true value.
- **RMSE = \$4,858** — The higher RMSE compared to MAE suggests some large errors on edge cases (very high-cost smokers).


In [ ]:
# ================================
# 04. Evaluation Visualisation
# ================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test_orig, y_pred, alpha=0.4, color='steelblue')
axes[0].plot([0, 65000], [0, 65000], 'r--', linewidth=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Charges ($)'); axes[0].set_ylabel('Predicted Charges ($)')
axes[0].set_title('Actual vs Predicted Charges'); axes[0].legend()

residuals = y_test_orig - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.4, color='coral')
axes[1].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Predicted Charges ($)'); axes[1].set_ylabel('Residuals ($)')
axes[1].set_title('Residual Plot')

plt.suptitle(f'XGBoost Model Performance | R² = {r2:.4f} | MAE = ${mae:,.0f}',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


**Interpretation:**
- The **Actual vs Predicted** plot shows points closely following the diagonal reference line, confirming good overall fit.
- The **Residual Plot** shows residuals mostly centred around zero with some variance at higher predicted values — expected for insurance data with extreme high-cost outliers.


In [ ]:
# ================================
# 05. Save Model
# ================================

os.makedirs('../models', exist_ok=True)
joblib.dump(model, '../models/xgboost_model.joblib')
joblib.dump(X_train.columns.tolist(), '../models/feature_names.joblib')
print("Model saved to ../models/")
print(f"Features: {X_train.columns.tolist()}")


---

# Part 3 — Explainable AI (XAI) Analysis

## Motivation

The XGBoost model trained in Part 2 achieves strong predictive performance (R² = 0.848). However, a black-box model alone is insufficient for deployment in a regulated industry like insurance.

**Two key questions remain:**
1. **Global:** Which features does the model rely on most across all customers?
2. **Local:** Why did the model assign a specific premium to a specific customer?

We apply five XAI methods to answer these questions — four covered in class, plus one bonus method not covered in class.

| Method | Scope | Source |
|---|---|---|
| Permutation Feature Importance | Global | In-class |
| PDP (Partial Dependence Plots) | Global | In-class |
| SHAP | Global + Local | In-class |
| LIME | Local | In-class |
| **Counterfactual Explanation** | Local | **Bonus — not covered in class** |


In [ ]:
# ================================
# 01. Imports & Setup
# ================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import joblib
from sklearn.preprocessing import LabelEncoder
from sklearn.inspection import PartialDependenceDisplay, permutation_importance
from sklearn.model_selection import train_test_split
from lime.lime_tabular import LimeTabularExplainer
import warnings
warnings.filterwarnings('ignore')

model = joblib.load('../models/xgboost_model.joblib')
feature_names = joblib.load('../models/feature_names.joblib')

df = pd.read_csv('../data/insurance.csv')
df_model = df.copy()
df_model['sex'] = LabelEncoder().fit_transform(df_model['sex'])
df_model['smoker'] = LabelEncoder().fit_transform(df_model['smoker'])
df_model = pd.get_dummies(df_model, columns=['region'], drop_first=True)

X = df_model[feature_names]
y = np.log1p(df_model['charges'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

explainer = shap.Explainer(model)
shap_values = explainer(X)

smoker_idx = df_model[df_model['smoker'] == 1].index[0]
non_smoker_idx = df_model[df_model['smoker'] == 0].index[0]

print("Setup complete. SHAP values computed.")
print(f"Data shape: {X.shape}")


## 3.1 Permutation Feature Importance

**What it is:** Permutation importance measures how much model performance decreases when a single feature's values are randomly shuffled (breaking any relationship between that feature and the target). A large drop in R² indicates high importance.

**Why it matters:** Unlike built-in tree importance (which can be biased), permutation importance is model-agnostic and directly tied to predictive performance.


In [ ]:
# ================================
# 02. Permutation Feature Importance
# ================================

result = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42, scoring='r2')

perm_df = pd.DataFrame({
    'feature': X.columns,
    'importance': result.importances_mean,
    'std': result.importances_std
}).sort_values('importance', ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(perm_df['feature'], perm_df['importance'],
         xerr=perm_df['std'], color='steelblue', edgecolor='white')
plt.xlabel('Decrease in R² Score')
plt.title('Permutation Feature Importance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(perm_df.sort_values('importance', ascending=False).to_string())


**Finding:** `smoker` and `age` cause the largest drops in R² when permuted, confirming they are the most critical predictors. `sex` and all `region` variables show near-zero importance — consistent with the EDA correlation findings.


## 3.2 Partial Dependence Plots (PDP)

**What it is:** A PDP shows the marginal effect of one feature on the predicted outcome, averaging over all other features. It reveals whether the relationship is linear, non-linear, or step-like.

**Interpretation guide:**  
- X-axis: feature value range  
- Y-axis: average predicted log(charges)  
- An upward slope = increasing the feature increases the predicted premium


In [ ]:
# ================================
# 03. Partial Dependence Plots
# ================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
features = ['age', 'bmi', 'smoker']

for i, feature in enumerate(features):
    feature_idx = list(X.columns).index(feature)
    PartialDependenceDisplay.from_estimator(
        model, X, [feature_idx], ax=axes[i],
        line_kw={'color': 'crimson', 'linewidth': 2}
    )
    axes[i].set_title(f'PDP: {feature}', fontweight='bold')
    axes[i].set_xlabel(feature)
    axes[i].set_ylabel('Predicted log(Charges)')

plt.suptitle('Partial Dependence Plots — How Features Affect Predictions',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


**Findings:**
- **Age:** Clear positive trend — premiums rise steadily as customers age. Legitimate actuarial logic.
- **BMI:** Non-linear relationship — premiums increase notably above BMI ~30 (clinical obesity threshold). This also reflects genuine health risk.
- **Smoker:** Near-step function — the transition from non-smoker (0) to smoker (1) produces a dramatic increase in predicted premiums. This is the single most impactful feature.


## 3.3 SHAP — Global Feature Importance

**What it is:** SHAP (SHapley Additive exPlanations) decomposes each prediction into contributions from each feature, grounded in cooperative game theory. Unlike simple feature importance, SHAP values have a consistent unit interpretation: they represent the change in predicted output attributable to each feature.

**Global view:** The bar chart shows mean |SHAP| values across all predictions — this is a fair, unbiased global feature importance measure.


In [ ]:
# ================================
# 04. SHAP — Global Feature Importance
# ================================

plt.figure(figsize=(10, 6))
shap.plots.bar(shap_values, max_display=8, show=False)
plt.title('Global Feature Importance (SHAP)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


**Finding:** `smoker` (SHAP = 0.48) is the dominant feature — more than twice the influence of `age` (0.41). `sex` (0.04) and all region variables (0.02–0.04) have near-zero global importance.

This confirms: **the model assigns premiums based on actual health risk factors, not uncontrollable demographic traits.**


## 3.4 SHAP — Beeswarm Plot

**What it is:** The beeswarm plot extends the bar chart by showing the distribution of SHAP values for each feature. Each dot represents one customer, coloured by feature value (red = high, blue = low).

This allows us to see not just *how important* a feature is, but *in which direction* and *how consistently* it affects predictions.


In [ ]:
# ================================
# 05. SHAP — Beeswarm Plot
# ================================

plt.figure(figsize=(10, 6))
shap.plots.beeswarm(shap_values, max_display=8, show=False)
plt.title('SHAP Beeswarm Plot — Feature Impact Distribution',
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


**Interpretation:**
- **Smoker (red dots far right):** Customers with `smoker=1` have large positive SHAP values — smoking dramatically increases premiums. The effect is highly consistent across all smokers.
- **Age:** Gradual spread — higher age (red) tends to push predictions higher, lower age (blue) pulls them lower.
- **Sex and Region:** Dots are clustered tightly around zero, confirming negligible and inconsistent influence.


## 3.5 SHAP — Local Explanation (Waterfall Plot)

**What it is:** A Waterfall plot provides an individual-level explanation — it shows exactly how each feature contributed to a specific customer's predicted premium, starting from the base value (average prediction) and building up to the final output.

**Business relevance:** This is the type of explanation that could be shown directly to a customer to justify their premium — transparent, auditable, and legally defensible.


In [ ]:
# ================================
# 06. SHAP — Individual Customer Explanation
# ================================

plt.figure(figsize=(10, 6))
shap.plots.waterfall(shap_values[smoker_idx], show=False)
plt.title('Individual Explanation — Smoker Customer',
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nCustomer Profile:")
print(f"  Actual charges: ${df.iloc[smoker_idx]['charges']:,.0f}")
print(f"  Smoker: {df.iloc[smoker_idx]['smoker']}")
print(f"  Age:    {df.iloc[smoker_idx]['age']}")
print(f"  BMI:    {df.iloc[smoker_idx]['bmi']}")


**Reading the chart:** Starting from the base value E[f(X)] = 9.107 (average log-premium across all customers), each bar shows a feature's push up (+) or pull down (-). The final value f(x) = 9.73 corresponds to a predicted premium of ~\$16,800.

For this 19-year-old smoker, `smoker=1` contributes +1.42 alone — dwarfing all other features combined.


In [ ]:
# ================================
# 07. SHAP — Non-Smoker vs Smoker Comparison
# ================================

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

plt.sca(axes[0])
shap.plots.waterfall(shap_values[non_smoker_idx], show=False)
axes[0].set_title(f'Non-Smoker | Actual: ${df.iloc[non_smoker_idx]["charges"]:,.0f}',
                   fontweight='bold', color='steelblue')

plt.sca(axes[1])
shap.plots.waterfall(shap_values[smoker_idx], show=False)
axes[1].set_title(f'Smoker | Actual: ${df.iloc[smoker_idx]["charges"]:,.0f}',
                   fontweight='bold', color='crimson')

plt.suptitle('SHAP Local Explanation: Non-Smoker vs Smoker',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


**Comparison:** The side-by-side waterfall charts show that the \$15,000+ difference in premiums between the two otherwise comparable customers is almost entirely explained by the `smoker` feature. This provides a clear, individual-level justification for the pricing difference.


## 3.6 LIME — Local Interpretable Model-Agnostic Explanations

**What it is:** LIME explains individual predictions by fitting a simple interpretable model (linear regression) locally around the specific data point of interest. Unlike SHAP, LIME is model-agnostic and does not require access to model internals.

**Why use both SHAP and LIME?** Using two independent local explanation methods provides cross-validation of our findings. If both methods agree on the most important features, we have higher confidence in the explanation.


In [ ]:
# ================================
# 08. LIME — Local Explanation
# ================================

lime_explainer = LimeTabularExplainer(
    training_data=X.values,
    feature_names=X.columns.tolist(),
    mode='regression'
)

exp = lime_explainer.explain_instance(
    data_row=X.iloc[smoker_idx].values,
    predict_fn=model.predict,
    num_features=8
)

fig = exp.as_pyplot_figure()
plt.title(f'LIME Explanation — Smoker Customer (Actual: ${df.iloc[smoker_idx]["charges"]:,.0f})',
          fontweight='bold')
plt.tight_layout()
plt.show()

print("\nLIME Feature Contributions:")
for feat, val in exp.as_list():
    print(f"  {feat}: {val:.4f}")


**Finding:** LIME confirms the SHAP result — `smoker > 0` has the largest positive contribution (+1.48), with `sex` and `region` variables contributing near zero. This cross-method consistency strengthens our conclusion.


## 3.7 Counterfactual Explanation *(Bonus — not covered in class)*

**What it is:** Counterfactual explanation answers the question: *"What would need to change about this customer for the model to predict a different outcome?"*

Unlike SHAP or LIME which explain past predictions, counterfactuals are **actionable** — they suggest what a customer (or insurer) could do differently.

**Business application:** In the insurance context, counterfactuals directly enable:
- Customer-facing conversations: *"If you quit smoking, your premium would drop by X"*
- Product design: Premium discount programmes linked to behavioural change
- Regulatory compliance: Demonstrating that pricing differences are tied to controllable, verifiable factors

**Implementation:** We construct a simple counterfactual by modifying the `smoker` feature from 1 → 0 while keeping all other features identical.


In [ ]:
# ================================
# 09. Counterfactual Analysis (Bonus)
# What if this customer quit smoking?
# ================================

original = X.iloc[smoker_idx].copy()
original_pred = np.expm1(model.predict(original.values.reshape(1, -1))[0])

counterfactual = original.copy()
counterfactual['smoker'] = 0
cf_pred = np.expm1(model.predict(counterfactual.values.reshape(1, -1))[0])

savings = original_pred - cf_pred

print("=== Counterfactual Analysis ===")
print(f"Customer: Age {df.iloc[smoker_idx]['age']}, BMI {df.iloc[smoker_idx]['bmi']}")
print(f"\nScenario 1 — Current (Smoker):   ${original_pred:,.0f}")
print(f"Scenario 2 — If Quit Smoking:    ${cf_pred:,.0f}")
print(f"Potential Savings:               ${savings:,.0f} per year")
print(f"Reduction:                       {savings/original_pred*100:.1f}%")

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(['Current\n(Smoker)', 'If Quit\nSmoking'],
               [original_pred, cf_pred],
               color=['crimson', 'steelblue'], edgecolor='white', width=0.4)
ax.bar_label(bars, fmt='$%.0f', padding=5, fontsize=12, fontweight='bold')
ax.set_ylabel('Predicted Insurance Charges ($)')
ax.set_title(f'Counterfactual: What If This Customer Quit Smoking?\nPotential Annual Savings: ${savings:,.0f} ({savings/original_pred*100:.1f}% reduction)',
             fontsize=12, fontweight='bold')
ax.set_ylim(0, original_pred * 1.3)
ax.annotate(f'Save ${savings:,.0f}!',
            xy=(1, cf_pred), xytext=(0.5, (original_pred+cf_pred)/2),
            fontsize=14, color='green', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='green'))
plt.tight_layout()
plt.show()


**Result:** If this smoker customer were to quit smoking, their predicted annual premium would drop from ~\$16,812 to ~\$2,131 — a saving of **\$14,681 (87.3% reduction)**.

This directly supports the business recommendation: a quit-smoking incentive programme benefits both the customer (lower premium) and the insurer (lower expected claims payout).


---

# Summary & Conclusions

## XAI Methods Applied

| Method | Type | Key Finding |
|---|---|---|
| Permutation Importance | Global | `smoker` and `age` cause largest performance drop when permuted |
| PDP | Global | Smoking creates a step-change; age/BMI show smooth positive trends |
| SHAP (bar + beeswarm) | Global | `smoker` SHAP = 0.48 — dominates all other features |
| SHAP (waterfall) | Local | Individual premiums largely explained by smoking status |
| LIME | Local | Independently confirms SHAP — `smoker` is the #1 local driver |
| **Counterfactual** *(Bonus)* | **Local** | **Quitting smoking saves \$14,681/year (87.3% reduction)** |

## Research Question — Answered

> *Are AI-driven insurance premiums based on legitimate health risk factors, or do they depend on uncontrollable personal traits?*

**Answer:** Our XGBoost model is driven primarily by `smoker` (SHAP = 0.48), `age` (0.41), and `bmi` (0.12) — all legitimate, health-related risk factors. In contrast, `sex` (0.04) and `region` (0.02–0.04) have near-zero influence. The model's pricing decisions are **fair and defensible**.

## Business Implications

1. **Regulatory compliance (ESG):** XAI provides technical proof that sex and region do not drive pricing — demonstrable GDPR and anti-discrimination compliance.
2. **Quit-smoking programme:** Counterfactual analysis quantifies the exact premium reduction from quitting — enables data-driven incentive programme design.
3. **Streamlined underwriting:** Since sex and region add negligible predictive value, they can be removed from intake forms — improving customer experience and cutting data costs.
